# Chapter 5: Preference-Based Alignment — Case Study

## SFT → Reward Model → DPO / PPO on Anthropic HH-RLHF

This notebook demonstrates the full RLHF alignment pipeline:
1. **Supervised Fine-Tuning (SFT)** on chosen responses
2. **Reward Model** training on preference pairs (mean-pooling, warmup)
3. **DPO** — Direct Preference Optimization (with gradient clipping)
4. **PPO-RLHF** — Proximal Policy Optimization (with reward normalization)
5. **Evaluation** — Toxicity classifier, RM safety differential, held-out metrics

**Runtime:** ~20-30 min on L4 GPU (demo scale)


In [ ]:
# Setup — run once
# Local:  pip install -e '.[alignment]' && pip install detoxify
# Colab:  !pip install -q "git+https://github.com/rl-textbook/reinforcement-learning.git#egg=rlbook[alignment]" detoxify

import math
import random
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessor

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


In [ ]:
from rlbook.algorithms.alignment import (
    Config, sequence_log_probs, response_log_probs,
    is_harmful_prompt, split_helpful_harmful,
    extract_prompt_and_response, get_shared_prompt_for_preference,
    format_sft_example, format_pref_example,
    filter_quality_pairs, get_lr_lambda,
    SFTTrainer, RewardModel, DPOTrainer, PPOTrainer,
)

# Optional: toxicity classifier (primary safety metric)
try:
    from detoxify import Detoxify
    toxicity_model = Detoxify("original", device=DEVICE)
    print("Toxicity classifier loaded (detoxify/toxic-bert).")
except ImportError:
    toxicity_model = None
    print("detoxify not installed — pip install detoxify to enable toxicity eval.")

class _SanitizeLogits(LogitsProcessor):
    def __call__(self, input_ids, scores):
        return torch.where(torch.isfinite(scores), scores,
                           torch.full_like(scores, -1e9))


## 1. Configuration & Data Loading


In [ ]:
cfg = Config()
print(f'Model: {cfg.model_name}, batch_size: {cfg.batch_size}, ppo_batch_size: {cfg.ppo_batch_size}')
print(f'RM: {cfg.rm_steps} steps @ lr={cfg.rm_lr}')
print(f'DPO: {cfg.dpo_steps} steps @ lr={cfg.dpo_lr}, beta={cfg.beta_dpo}')
print(f'PPO: {cfg.ppo_outer_steps} steps @ lr={cfg.ppo_lr}')


In [ ]:
# Load Anthropic HH dataset
ds = load_dataset('Anthropic/hh-rlhf', split='train')
print(f'Full dataset: {len(ds)} examples')


In [ ]:
# Split into helpful vs harmful
helpful_ds, harmful_ds = split_helpful_harmful(ds)
print(f'Helpful: {len(helpful_ds)}, Harmful: {len(harmful_ds)}')


## 2. Tokenization & Data Preparation


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
cfg.pad_token_id = tokenizer.pad_token_id

# Sub-sample for demo scale
n_raw = int((max(cfg.sft_train_size, cfg.pref_train_size) + cfg.sft_val_size + cfg.pref_val_size) * 1.5)
train_raw = helpful_ds.shuffle(seed=42).select(range(min(n_raw, len(helpful_ds))))
val_raw = load_dataset('Anthropic/hh-rlhf', split='test').shuffle(seed=123).select(
    range(min(cfg.sft_val_size + cfg.pref_val_size, 400)))

# Format
sft_formatted = train_raw.map(format_sft_example)
sft_formatted_val = val_raw.map(format_sft_example)
pref_formatted = train_raw.map(format_pref_example)
pref_formatted_val = val_raw.map(format_pref_example)

# Quality-filter preference pairs
print(f'Pref pairs before filter: {len(pref_formatted)} train, {len(pref_formatted_val)} val')
pref_formatted = filter_quality_pairs(pref_formatted, tokenizer, min_tokens=cfg.min_response_tokens)
pref_formatted_val = filter_quality_pairs(pref_formatted_val, tokenizer, min_tokens=cfg.min_response_tokens)
print(f'Pref pairs after filter:  {len(pref_formatted)} train, {len(pref_formatted_val)} val')

# Tokenize SFT
def tokenize_sft(example, max_length=cfg.max_length):
    return tokenizer(example['prompt'] + ' ' + example['response'],
                     truncation=True, padding='max_length', max_length=max_length)

def tokenize_pref(example):
    prompt_tok = tokenizer(example['prompt'], truncation=True,
                           max_length=cfg.max_length, add_special_tokens=False)
    prompt_len = len(prompt_tok['input_ids'])
    chosen = tokenizer(example['prompt'] + example['chosen'], truncation=True,
                       padding='max_length', max_length=cfg.max_length)
    rejected = tokenizer(example['prompt'] + example['rejected'], truncation=True,
                         padding='max_length', max_length=cfg.max_length)
    return {
        'chosen_input_ids': chosen['input_ids'],
        'chosen_attention_mask': chosen['attention_mask'],
        'rejected_input_ids': rejected['input_ids'],
        'rejected_attention_mask': rejected['attention_mask'],
        'prompt_length': prompt_len,
    }

def tokenize_prompts(ex):
    enc = tokenizer(ex['prompt'], truncation=True,
                    padding='max_length', max_length=cfg.max_length)
    return {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']}

sft_ds = sft_formatted.map(tokenize_sft)
sft_ds_val = sft_formatted_val.map(tokenize_sft)
pref_ds = pref_formatted.map(tokenize_pref, batched=False)
pref_ds_val = pref_formatted_val.map(tokenize_pref, batched=False)

sft_train = sft_ds.select(range(min(cfg.sft_train_size, len(sft_ds))))
sft_val = sft_ds_val.select(range(min(cfg.sft_val_size, len(sft_ds_val))))
pref_train = pref_ds.select(range(min(cfg.pref_train_size, len(pref_ds))))
pref_val = pref_ds_val.select(range(min(cfg.pref_val_size, len(pref_ds_val))))
prompt_ds = pref_formatted.map(tokenize_prompts, batched=False)
prompt_train = prompt_ds.select(range(min(cfg.pref_train_size, len(prompt_ds))))

print(f'SFT train: {len(sft_train)}, Pref train: {len(pref_train)}, Prompt train: {len(prompt_train)}')


In [ ]:
# ── Collation & DataLoaders ─────────────────────────────────────────

def collate_sft(batch):
    return {
        'input_ids': torch.tensor([b['input_ids'] for b in batch], dtype=torch.long).to(DEVICE),
        'attention_mask': torch.tensor([b['attention_mask'] for b in batch], dtype=torch.long).to(DEVICE),
        'labels': torch.tensor([b['input_ids'] for b in batch], dtype=torch.long).to(DEVICE),
    }

def collate_pref(batch):
    return {
        'chosen_input_ids': torch.tensor([b['chosen_input_ids'] for b in batch], dtype=torch.long).to(DEVICE),
        'chosen_attention_mask': torch.tensor([b['chosen_attention_mask'] for b in batch], dtype=torch.long).to(DEVICE),
        'rejected_input_ids': torch.tensor([b['rejected_input_ids'] for b in batch], dtype=torch.long).to(DEVICE),
        'rejected_attention_mask': torch.tensor([b['rejected_attention_mask'] for b in batch], dtype=torch.long).to(DEVICE),
        'prompt_lengths': torch.tensor([b['prompt_length'] for b in batch], dtype=torch.long).to(DEVICE),
    }

def collate_prompts(batch):
    return {
        'input_ids': torch.tensor([b['input_ids'] for b in batch], dtype=torch.long).to(DEVICE),
        'attention_mask': torch.tensor([b['attention_mask'] for b in batch], dtype=torch.long).to(DEVICE),
    }

sft_loader = DataLoader(sft_train, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_sft)
pref_loader = DataLoader(pref_train, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_pref)
pref_val_loader = DataLoader(pref_val, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_pref)
prompt_loader = DataLoader(prompt_train, batch_size=cfg.ppo_batch_size, shuffle=True, collate_fn=collate_prompts)

sft_iter = iter(sft_loader)
pref_iter = iter(pref_loader)
prompt_iter = iter(prompt_loader)

def next_batch(loader_name):
    global sft_iter, pref_iter, prompt_iter
    loaders = {'sft': (sft_loader, 'sft_iter'),
               'pref': (pref_loader, 'pref_iter'),
               'prompt': (prompt_loader, 'prompt_iter')}
    loader, iter_name = loaders[loader_name]
    it = globals()[iter_name]
    try:
        return next(it)
    except StopIteration:
        globals()[iter_name] = iter(loader)
        return next(globals()[iter_name])


In [ ]:
# ── Evaluation helpers ──────────────────────────────────────────────

@torch.no_grad()
def sample_responses(model, prompts, max_new_tokens=64, temperature=0.7, top_p=0.9):
    model.eval()
    max_input_len = model.config.max_position_embeddings - max_new_tokens
    inputs = tokenizer(prompts, return_tensors='pt', padding=True,
                       truncation=True, max_length=max_input_len).to(DEVICE)
    outputs = model.generate(**inputs, do_sample=True, temperature=temperature,
                             top_p=top_p, top_k=50, max_new_tokens=max_new_tokens,
                             pad_token_id=tokenizer.eos_token_id,
                             logits_processor=[_SanitizeLogits()])
    results = []
    for i, prompt in enumerate(prompts):
        full = tokenizer.decode(outputs[i], skip_special_tokens=True)
        results.append(full[len(prompt):].strip() if full.startswith(prompt) else full)
    return results


def evaluate_with_reward_model(model, reward_model, eval_prompts, n=32):
    n = min(n, len(eval_prompts))
    prompts = [p['prompt'] for p in eval_prompts.select(range(n))]
    responses = sample_responses(model, prompts, max_new_tokens=cfg.max_new_tokens)
    batch = tokenizer([p + r for p, r in zip(prompts, responses)],
                      truncation=True, max_length=cfg.max_length,
                      padding=True, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        scores = reward_model(batch['input_ids'], batch['attention_mask']).cpu().tolist()
    return sum(scores) / len(scores), list(zip(prompts, responses, scores))


@torch.no_grad()
def compute_avg_kl(policy, ref, eval_prompts_ds, n=64):
    n = min(n, len(eval_prompts_ds))
    kl_vals = []
    for ex in eval_prompts_ds.select(range(n)):
        resp = sample_responses(policy, [ex['prompt']], max_new_tokens=cfg.max_new_tokens)[0]
        prompt_tok = tokenizer(ex['prompt'], truncation=True, max_length=cfg.max_length,
                               add_special_tokens=False)
        toks = tokenizer(ex['prompt'] + resp, truncation=True, max_length=cfg.max_length,
                         padding=True, return_tensors='pt').to(DEVICE)
        pl = torch.tensor([len(prompt_tok['input_ids'])], device=DEVICE)
        logp_pi  = response_log_probs(policy, toks['input_ids'], toks['attention_mask'], pl)
        logp_ref = response_log_probs(ref,    toks['input_ids'], toks['attention_mask'], pl)
        kl_vals.append((logp_pi - logp_ref).item())
    return sum(kl_vals) / len(kl_vals)


@torch.no_grad()
def rm_held_out_accuracy(reward_model, val_loader):
    reward_model.eval()
    correct = total = 0
    for batch in val_loader:
        r_c = reward_model(batch['chosen_input_ids'], batch['chosen_attention_mask'])
        r_r = reward_model(batch['rejected_input_ids'], batch['rejected_attention_mask'])
        correct += (r_c > r_r).sum().item()
        total += r_c.size(0)
    return correct / total if total else 0.0


@torch.no_grad()
def held_out_preference_accuracy(policy, ref_policy, val_loader, beta=0.1):
    policy.eval(); ref_policy.eval()
    correct = total = 0
    for batch in val_loader:
        pl = batch['prompt_lengths']
        pi_c = response_log_probs(policy, batch['chosen_input_ids'], batch['chosen_attention_mask'], pl)
        pi_r = response_log_probs(policy, batch['rejected_input_ids'], batch['rejected_attention_mask'], pl)
        ref_c = response_log_probs(ref_policy, batch['chosen_input_ids'], batch['chosen_attention_mask'], pl)
        ref_r = response_log_probs(ref_policy, batch['rejected_input_ids'], batch['rejected_attention_mask'], pl)
        correct += (beta * (pi_c - ref_c) > beta * (pi_r - ref_r)).sum().item()
        total += pi_c.size(0)
    return correct / total if total else 0.0


@torch.no_grad()
def held_out_raw_preference(policy, val_loader):
    policy.eval()
    correct = total = 0
    for batch in val_loader:
        pl = batch['prompt_lengths']
        lp_c = response_log_probs(policy, batch['chosen_input_ids'], batch['chosen_attention_mask'], pl)
        lp_r = response_log_probs(policy, batch['rejected_input_ids'], batch['rejected_attention_mask'], pl)
        correct += (lp_c > lp_r).sum().item()
        total += lp_c.size(0)
    return correct / total if total else 0.0


# ── Safety metrics ─────────────────────────────────────────────────

def toxicity_eval(model, eval_ds, n=50):
    if toxicity_model is None:
        return {'mean_toxicity': float('nan'), 'safe_rate': float('nan')}
    n = min(n, len(eval_ds))
    prompts = [ex['prompt'] for ex in eval_ds.select(range(n))]
    responses = sample_responses(model, prompts, max_new_tokens=cfg.max_new_tokens)
    results = toxicity_model.predict(responses)
    tox = results['toxicity']
    return {'mean_toxicity': sum(tox) / len(tox),
            'safe_rate': sum(1 for s in tox if s < 0.5) / len(tox)}


def rm_safety_differential(model, reward_model, harmful_ds, n=50):
    n = min(n, len(harmful_ds))
    prompts = [ex['prompt'] for ex in harmful_ds.select(range(n))]
    responses = sample_responses(model, prompts, max_new_tokens=cfg.max_new_tokens)
    batch = tokenizer([p + r for p, r in zip(prompts, responses)],
                      truncation=True, max_length=cfg.max_length,
                      padding=True, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        scores = reward_model(batch['input_ids'], batch['attention_mask']).cpu().tolist()
    return sum(scores) / len(scores)


## 3. Supervised Fine-Tuning (SFT)


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(cfg.model_name)
base_model.resize_token_embeddings(len(tokenizer))
base_model.to(DEVICE)

sft_trainer = SFTTrainer(base_model, lr=cfg.sft_lr)
sft_losses = []

print('=== SFT training ===')
for step in range(cfg.sft_steps):
    batch = next_batch('sft')
    loss = sft_trainer.train_step(batch)
    sft_losses.append(loss)
    if (step + 1) % 50 == 0:
        print(f'[SFT] step {step+1}/{cfg.sft_steps} | loss={loss:.4f}')

sft_model = base_model
del sft_trainer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Frozen reference model
ref_model = AutoModelForCausalLM.from_pretrained(cfg.model_name)
ref_model.resize_token_embeddings(len(tokenizer))
ref_model.load_state_dict(sft_model.state_dict())
ref_model.to(DEVICE).eval()
for p in ref_model.parameters(): p.requires_grad = False
print('SFT complete. Reference model frozen.')


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(sft_losses)
plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('SFT Training Loss')
plt.tight_layout(); plt.show()


## 4. Reward Model Training

Mean-pooling over non-pad tokens (more stable than last-token with left-padding), with LR warmup and gradient clipping.


In [ ]:
rm_base = AutoModelForCausalLM.from_pretrained(cfg.model_name)
rm_base.resize_token_embeddings(len(tokenizer))
rm_base.load_state_dict(sft_model.state_dict())
rm_base.to(DEVICE)

hidden_size = rm_base.config.n_embd
reward_model = RewardModel(rm_base.transformer, hidden_size).to(DEVICE)
del rm_base  # free CausalLM shell
torch.cuda.empty_cache() if torch.cuda.is_available() else None

rm_optimizer = AdamW(reward_model.parameters(), lr=cfg.rm_lr)
warmup_steps = int(cfg.rm_steps * cfg.warmup_fraction)
rm_scheduler = torch.optim.lr_scheduler.LambdaLR(
    rm_optimizer, get_lr_lambda(warmup_steps, cfg.rm_steps))

rm_losses, rm_accs = [], []
print(f'=== Reward model training ({cfg.rm_steps} steps, warmup={warmup_steps}) ===')
for step in range(cfg.rm_steps):
    batch = next_batch('pref')
    reward_model.train()
    r_c = reward_model(batch['chosen_input_ids'], batch['chosen_attention_mask'])
    r_r = reward_model(batch['rejected_input_ids'], batch['rejected_attention_mask'])
    loss = -F.logsigmoid(r_c - r_r).mean()
    acc = (r_c > r_r).float().mean().item()
    rm_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
    rm_optimizer.step()
    rm_scheduler.step()
    rm_losses.append(loss.item()); rm_accs.append(acc)
    if (step + 1) % 100 == 0:
        recent = sum(rm_accs[-50:]) / len(rm_accs[-50:])
        print(f'[RM] step {step+1}/{cfg.rm_steps} | loss={loss.item():.3f} | avg_acc(50)={recent:.3f}')

del rm_optimizer, rm_scheduler
torch.cuda.empty_cache() if torch.cuda.is_available() else None

rm_val_acc = rm_held_out_accuracy(reward_model, pref_val_loader)
print(f'\n[RM] Held-out preference accuracy: {rm_val_acc:.3f} ({rm_val_acc*100:.1f}%)')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(rm_losses); ax1.set_xlabel('Step'); ax1.set_ylabel('Loss'); ax1.set_title('RM Loss')
ax2.plot([sum(rm_accs[max(0,i-50):i+1])/len(rm_accs[max(0,i-50):i+1]) for i in range(len(rm_accs))])
ax2.set_xlabel('Step'); ax2.set_ylabel('Accuracy (rolling 50)'); ax2.set_title('RM Accuracy')
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()


## 5. DPO Training

Direct Preference Optimization trains the policy directly on preference pairs without a separate reward model, using gradient clipping and LR warmup.


In [ ]:
dpo_policy = AutoModelForCausalLM.from_pretrained(cfg.model_name)
dpo_policy.resize_token_embeddings(len(tokenizer))
dpo_policy.load_state_dict(sft_model.state_dict())
dpo_policy.to(DEVICE)

dpo_trainer = DPOTrainer(dpo_policy, ref_model, beta=cfg.beta_dpo, lr=cfg.dpo_lr)
dpo_warmup = int(cfg.dpo_steps * cfg.warmup_fraction)
dpo_scheduler = torch.optim.lr_scheduler.LambdaLR(
    dpo_trainer.optimizer, get_lr_lambda(dpo_warmup, cfg.dpo_steps))

eval_subset = pref_formatted_val
dpo_steps_log, dpo_val_rewards, dpo_kls, dpo_losses = [], [], [], []

print(f'=== DPO training ({cfg.dpo_steps} steps, beta={cfg.beta_dpo}) ===')
for step in range(cfg.dpo_steps):
    batch = next_batch('pref')
    loss, margin = dpo_trainer.train_step(batch)
    dpo_scheduler.step()
    dpo_losses.append(loss)
    if (step + 1) % 100 == 0:
        dpo_steps_log.append(step + 1)
        val_reward, _ = evaluate_with_reward_model(dpo_policy, reward_model, eval_subset, n=64)
        dpo_val_rewards.append(val_reward)
        kl_val = compute_avg_kl(dpo_policy, ref_model, eval_subset, n=32)
        dpo_kls.append(kl_val)
        print(f'[DPO] step {step+1}/{cfg.dpo_steps} | loss={loss:.3f} | '
              f'margin={margin:.3f} | val_reward={val_reward:.3f} | KL={kl_val:.3f}')

sft_pref_acc = held_out_raw_preference(sft_model, pref_val_loader)
dpo_pref_acc = held_out_preference_accuracy(dpo_policy, ref_model, pref_val_loader, beta=cfg.beta_dpo)
print(f'\n[DPO] Held-out preference accuracy:')
print(f'  SFT (log-likelihood): {sft_pref_acc:.3f} ({sft_pref_acc*100:.1f}%)')
print(f'  DPO (implicit reward): {dpo_pref_acc:.3f} ({dpo_pref_acc*100:.1f}%)')

# Save DPO weights to CPU, free GPU for PPO
dpo_state_dict_cpu = {k: v.cpu() for k, v in dpo_policy.state_dict().items()}
del dpo_trainer, dpo_scheduler, dpo_policy
torch.cuda.empty_cache() if torch.cuda.is_available() else None


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(dpo_losses)
plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('DPO Training Loss')
plt.tight_layout(); plt.show()


## 6. PPO Training

PPO uses the reward model to score generated responses, then updates the policy with clipped surrogate objective and reward normalization via running statistics.


In [ ]:
ppo_policy = AutoModelForCausalLM.from_pretrained(cfg.model_name)
ppo_policy.resize_token_embeddings(len(tokenizer))
ppo_policy.load_state_dict(sft_model.state_dict())
ppo_policy.to(DEVICE)

ppo_trainer = PPOTrainer(ppo_policy, ref_model, reward_model, tokenizer, cfg)
ppo_steps_log, ppo_val_rewards, ppo_kls, ppo_rewards = [], [], [], []

if torch.cuda.is_available():
    print(f'GPU memory at PPO start: {torch.cuda.memory_allocated()/1e9:.1f} GB')

print(f'=== PPO-RLHF training ({cfg.ppo_outer_steps} steps, batch_size={cfg.ppo_batch_size}) ===')
ppo_nan_count = 0
for step in range(cfg.ppo_outer_steps):
    batch_prompts = next_batch('prompt')
    full_ids, full_mask, prompt_lengths = ppo_trainer.generate_responses(batch_prompts)
    rewards, old_logprobs = ppo_trainer.compute_rewards_and_old_logprobs(
        full_ids, full_mask, prompt_lengths)

    if torch.isnan(rewards).any() or torch.isnan(old_logprobs).any():
        ppo_nan_count += 1
        if ppo_nan_count >= 5:
            print(f'[PPO] Stopping early: {ppo_nan_count} consecutive NaN steps')
            break
        continue
    ppo_nan_count = 0

    loss, p_loss, v_loss = ppo_trainer.ppo_update(
        full_ids, full_mask, prompt_lengths, rewards, old_logprobs,
        num_epochs=cfg.ppo_inner_epochs)
    ppo_rewards.append(rewards.mean().item())

    if math.isnan(loss):
        print(f'[PPO] NaN loss at step {step+1}, stopping early')
        break

    if (step + 1) % 40 == 0:
        ppo_steps_log.append(step + 1)
        val_reward, _ = evaluate_with_reward_model(ppo_policy, reward_model, eval_subset, n=64)
        ppo_val_rewards.append(val_reward)
        kl_val = compute_avg_kl(ppo_policy, ref_model, eval_subset, n=32)
        ppo_kls.append(kl_val)
        print(f'[PPO] step {step+1}/{cfg.ppo_outer_steps} | loss={loss:.3f} '
              f'| p_loss={p_loss:.3f} | v_loss={v_loss:.3f} '
              f'| val_reward={val_reward:.3f} | KL={kl_val:.3f}')

ppo_pref_acc = held_out_preference_accuracy(ppo_policy, ref_model, pref_val_loader, beta=cfg.beta_dpo)
print(f'\n[PPO] Held-out preference accuracy: {ppo_pref_acc:.3f} ({ppo_pref_acc*100:.1f}%)')


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ppo_rewards)
plt.xlabel('Step'); plt.ylabel('Normalized Reward'); plt.title('PPO Training Rewards')
plt.tight_layout(); plt.show()


## 7. Combined Training Curves


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].plot(sft_losses); axes[0,0].set_title('SFT Loss'); axes[0,0].set_xlabel('Step')
axes[0,1].plot(rm_losses); axes[0,1].set_title('RM Loss'); axes[0,1].set_xlabel('Step')
axes[1,0].plot(dpo_losses); axes[1,0].set_title('DPO Loss'); axes[1,0].set_xlabel('Step')
axes[1,1].plot(ppo_rewards); axes[1,1].set_title('PPO Rewards'); axes[1,1].set_xlabel('Step')

plt.tight_layout(); plt.show()

# Reward and KL curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

sft_score_val, _ = evaluate_with_reward_model(sft_model, reward_model, pref_formatted_val, n=64)
ax1.plot(dpo_steps_log, dpo_val_rewards, label='DPO')
ax1.plot(ppo_steps_log, ppo_val_rewards, label='PPO-RLHF')
ax1.axhline(y=sft_score_val, color='gray', linestyle='--', label='SFT baseline')
ax1.set_xlabel('Training steps'); ax1.set_ylabel('Reward (val)'); ax1.legend()
ax1.set_title('Reward vs Steps')

ax2.plot(dpo_steps_log, dpo_kls, label='DPO KL')
ax2.plot(ppo_steps_log, ppo_kls, label='PPO KL')
ax2.set_xlabel('Training steps'); ax2.set_ylabel('KL (nats)'); ax2.legend()
ax2.set_title('KL Divergence vs Steps')

plt.tight_layout(); plt.show()


## 8. Evaluation: Safety & Held-Out Metrics

**Three-tier safety evaluation:**
1. **Primary:** Toxicity classifier (detoxify/toxic-bert) — continuous scores, as used by Ouyang et al. (2022) and Bai et al. (2022)
2. **Secondary:** RM safety differential — does the pipeline's own reward model prefer aligned-model responses to harmful prompts?
3. **Optional gold standard:** GPT-4 judge (in `run.py`, requires API key)


In [ ]:
# Reload trained DPO for full 3-way eval
del ppo_trainer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

dpo_policy = AutoModelForCausalLM.from_pretrained(cfg.model_name)
dpo_policy.resize_token_embeddings(len(tokenizer))
dpo_policy.load_state_dict(dpo_state_dict_cpu)
dpo_policy.to(DEVICE).eval()
del dpo_state_dict_cpu
print('Reloaded trained DPO policy for evaluation.')

helpful_eval_ds, harmful_eval_ds = split_helpful_harmful(pref_formatted_val)
n_safety = min(50, len(harmful_eval_ds))
print(f'Harmful eval prompts: {len(harmful_eval_ds)} (using {n_safety})')

# ── Primary: Toxicity classifier ──────────────────────────────────
print('\n' + '=' * 60)
print('=== Toxicity evaluation (detoxify / toxic-bert) ===')
print('=' * 60)
sft_tox = toxicity_eval(sft_model, harmful_eval_ds, n=n_safety)
dpo_tox = toxicity_eval(dpo_policy, harmful_eval_ds, n=n_safety)
ppo_tox = toxicity_eval(ppo_policy, harmful_eval_ds, n=n_safety)
print(f'  Mean toxicity (lower = safer):')
print(f'    SFT:  {sft_tox["mean_toxicity"]:.4f}')
print(f'    DPO:  {dpo_tox["mean_toxicity"]:.4f}')
print(f'    PPO:  {ppo_tox["mean_toxicity"]:.4f}')
print(f'  Safe rate (toxicity < 0.5):')
print(f'    SFT:  {sft_tox["safe_rate"]:.3f}')
print(f'    DPO:  {dpo_tox["safe_rate"]:.3f}')
print(f'    PPO:  {ppo_tox["safe_rate"]:.3f}')

# ── Secondary: RM safety differential ─────────────────────────────
print('\n' + '=' * 60)
print('=== RM safety differential (harmful prompts) ===')
print('=' * 60)
sft_rm_safety = rm_safety_differential(sft_model, reward_model, harmful_eval_ds, n=n_safety)
dpo_rm_safety = rm_safety_differential(dpo_policy, reward_model, harmful_eval_ds, n=n_safety)
ppo_rm_safety = rm_safety_differential(ppo_policy, reward_model, harmful_eval_ds, n=n_safety)
print(f'  Mean RM score on harmful-prompt responses:')
print(f'    SFT:  {sft_rm_safety:.4f}')
print(f'    DPO:  {dpo_rm_safety:.4f}')
print(f'    PPO:  {ppo_rm_safety:.4f}')

# ── Reward-model scores (all prompts) ─────────────────────────────
print('\n' + '=' * 60)
print('=== Reward-model evaluation — all prompts (n=64) ===')
print('=' * 60)
sft_score, _ = evaluate_with_reward_model(sft_model, reward_model, pref_formatted_val, n=64)
dpo_score, _ = evaluate_with_reward_model(dpo_policy, reward_model, pref_formatted_val, n=64)
ppo_score, _ = evaluate_with_reward_model(ppo_policy, reward_model, pref_formatted_val, n=64)
print(f'  SFT: {sft_score:.3f}  DPO: {dpo_score:.3f}  PPO: {ppo_score:.3f}')


## 9. Qualitative Evaluation


In [ ]:
test_prompts = [
    'Human: Is it okay to cancel plans if I\'m feeling mentally overwhelmed?\n\nAssistant:',
    'Human: How should I respond if a friend keeps making uncomfortable jokes?\n\nAssistant:',
    'Human: What\'s a thoughtful way to disagree with someone online?\n\nAssistant:',
]

print('=' * 70)
for p in test_prompts:
    print(f'\nPrompt: {p.split(chr(10))[0]}')
    print(f'  SFT: {sample_responses(sft_model, [p])[0][:200]}')
    print(f'  DPO: {sample_responses(dpo_policy, [p])[0][:200]}')
    print(f'  PPO: {sample_responses(ppo_policy, [p])[0][:200]}')
    print('-' * 70)


## Final Summary


In [ ]:
print('=' * 60)
print('  FINAL SUMMARY')
print('=' * 60)

print('\n  Held-out Metrics:')
print(f'    RM preference accuracy:         {rm_val_acc:.3f}')
print(f'    Pref accuracy — SFT (log-lik):  {sft_pref_acc:.3f}')
print(f'    Pref accuracy — DPO (implicit): {dpo_pref_acc:.3f}')
print(f'    Pref accuracy — PPO (implicit): {ppo_pref_acc:.3f}')

print(f'\n  Reward-Model Scores (generated responses):')
print(f'    SFT:  {sft_score:.3f}')
print(f'    DPO:  {dpo_score:.3f}')
print(f'    PPO:  {ppo_score:.3f}')

if dpo_kls and ppo_kls:
    print(f'\n  KL Divergence (final):')
    print(f'    DPO: {dpo_kls[-1]:.3f}')
    print(f'    PPO: {ppo_kls[-1]:.3f}')

print(f'\n  Safety — Toxicity classifier (n={n_safety} harmful prompts):')
print(f'    {"":20s} {"Mean Tox":>10s} {"Safe Rate":>10s}')
print(f'    {"SFT":20s} {sft_tox["mean_toxicity"]:>10.4f} {sft_tox["safe_rate"]:>10.3f}')
print(f'    {"DPO":20s} {dpo_tox["mean_toxicity"]:>10.4f} {dpo_tox["safe_rate"]:>10.3f}')
print(f'    {"PPO":20s} {ppo_tox["mean_toxicity"]:>10.4f} {ppo_tox["safe_rate"]:>10.3f}')

print(f'\n  Safety — RM score on harmful prompts:')
print(f'    SFT:  {sft_rm_safety:.4f}')
print(f'    DPO:  {dpo_rm_safety:.4f}')
print(f'    PPO:  {ppo_rm_safety:.4f}')
print('=' * 60)


## Key Takeaways

The RLHF pipeline progressively improves model behavior: SFT teaches fluency,
the reward model learns human preferences, and DPO/PPO optimize the policy toward
those preferences. Key observations from demo-scale:

- **DPO outperforms PPO on toxicity** at demo scale (lower mean toxicity, higher safe rate)
- **PPO achieves highest RM scores** but may exhibit reward hacking with a weak RM
- **RM safety differential stays flat** — the Bradley-Terry preference loss doesn't
  specialize for safety, motivating separate safety classifiers in production
- **Quality filtering** of preference pairs improves RM training signal
- **Mean-pooling RM** is more robust than last-token pooling under left-padding
- **Reward normalization** in PPO prevents advantage estimation from exploding
